# 05. The Attention Mechanism in Sequence-to-Sequence Models

In classical Seq2Seq encoder-decoder RNNs (Sutskever et al., 2014), the entire input sequence must be compressed into a single fixed-length context vector $\mathbf{c}$. For sequences longer than ~20 tokens, this creates an informational bottleneck.

### Bahdanau Additive Attention (2014):
Instead of a static context vector, the decoder dynamically computes an attention weight $\alpha_{ij}$ over all encoder hidden states $\mathbf{h}_i$ at each decoding step $j$:
$$e_{ij} = \mathbf{v}_a^T \tanh(\mathbf{W}_a \mathbf{s}_{j-1} + \mathbf{U}_a \mathbf{h}_i)$$
$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_k \exp(e_{ik})}, \quad \mathbf{c}_j = \sum_i \alpha_{ij} \mathbf{h}_i$$

In [ ]:
import torch
import torch.nn as nn

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_s = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (Batch, 1, Dim)
        # encoder_outputs: (Batch, SeqLen, Dim)
        scores = self.v(torch.tanh(self.W_s(decoder_hidden) + self.W_h(encoder_outputs)))
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.sum(attn_weights * encoder_outputs, dim=1, keepdim=True)
        return context, attn_weights

attn = BahdanauAttention(hidden_dim=32)
enc_out = torch.randn(2, 6, 32)
dec_hid = torch.randn(2, 1, 32)
ctx, weights = attn(dec_hid, enc_out)
print("Dynamic Context Shape: ", ctx.shape)
print("Attention Weights Shape:", weights.shape)

## 🎯 Exercises & Practice
1. Verify that attention weights sum to exactly 1.0 across the sequence dimension for each batch sample.

In [ ]:
# Exercise Solution
print("Attention Weight Sums:", weights.sum(dim=1).squeeze().detach().numpy())